## RQ-3: Generalisierbarkeit des Agenten

Dieses Notebook wertet aus, wie gut der Visualisierungsagent über verschiedene Konfigurationen hinweg generalisiert. Untersucht werden vier Konfigurationen, die sich nach Sprache (Deutsch/Englisch) und Zielsprache (Python/R) unterscheiden.

**Datenbasis:** 50 Datensätze pro Konfiguration (3 konnten nicht geladen werden, 47 wurden tatsächlich gestartet)

**Metriken:**
- **DC** (Dataset Coverage): Anteil der Datensätze, bei denen die Visualisierungsphase erreicht wurde
- **VER** (Visualization Error Rate): Anteil fehlgeschlagener Visualisierungen in Iteration 0
- **CER** (Convergence Error Rate): Anteil der Datensätze, die nach allen Iterationen nicht konvergierten
- **AIR** (Avg. Iterations Required): Mittlere Iterationsanzahl bis zur Konvergenz (nur konvergierte Datensätze)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
BASE_PATH = Path("./src/resources/paper_results/50_dataset_run")
STUDY_PATH = Path("./results/rq_3/study_data.csv")
OUTPUT_DIR = Path("./results/rq_3/figures")
THESIS_FIG_DIR = OUTPUT_DIR

TOTAL_DATASETS = 50

CONFIGS = {
    "DE+Python": BASE_PATH / "DE" / "Python" / "output",
    "DE+R":      BASE_PATH / "DE" / "R"      / "output",
    "EN+Python": BASE_PATH / "EN" / "Python" / "output",
    "EN+R":      BASE_PATH / "EN" / "R"      / "output",
}
CONFIG_ORDER = ["DE+Python", "DE+R", "EN+Python", "EN+R"]

COLORS = {
    "DE+Python": "#334152",   # HTWG Dark Blue
    "DE+R":      "#009B91",   # HTWG Teal
    "EN+Python": "#7990AB",   # HTWG Medium Blue
    "EN+R":      "#546B86",   # HTWG Shaded Blue
}

---
### Hilfsfunktion: Konfigurationsanalyse

Die folgende Funktion liest für eine gegebene Konfiguration alle Datensatz-Unterordner ein und berechnet die vier oben definierten Metriken. Sie gibt sowohl eine Zusammenfassung auf Konfigurationsebene als auch die Rohdaten pro Datensatz zurück.

In [ ]:
def analyse_config(config_name: str, output_path: Path) -> dict:
    dataset_dirs = [d for d in output_path.iterdir() if d.is_dir()]
    started = len(dataset_dirs)

    results_per_dataset = []

    for ds_dir in sorted(dataset_dirs):
        ver_files = sorted(ds_dir.glob("VER_*.csv"), key=lambda p: int(p.stem.split("_")[1]))

        if not ver_files:
            results_per_dataset.append({
                "dataset": ds_dir.name,
                "reached_viz": False,
                "n_iterations": 0,
                "n_visualizations": None,
                "ver_iter0": None,
                "converged": False,
                "n_success_final": None,
                "n_fail_final": None,
            })
            continue

        df0 = pd.read_csv(ver_files[0])
        n_vis = len(df0)
        n_fail_iter0 = (df0["ver_value"] == 0).sum()
        ver_iter0 = (n_fail_iter0 / n_vis) * 100 if n_vis > 0 else np.nan

        df_last = pd.read_csv(ver_files[-1])
        n_success_final = (df_last["ver_value"] == 1).sum()
        n_fail_final    = (df_last["ver_value"] == 0).sum()
        converged       = n_fail_final == 0

        results_per_dataset.append({
            "dataset": ds_dir.name,
            "reached_viz": True,
            "n_iterations": len(ver_files),
            "n_visualizations": n_vis,
            "ver_iter0": ver_iter0,
            "converged": converged,
            "n_success_final": int(n_success_final),
            "n_fail_final": int(n_fail_final),
        })

    df = pd.DataFrame(results_per_dataset)
    reached   = df["reached_viz"].sum()
    converged = df[df["reached_viz"]]["converged"].sum()
    not_converged = reached - converged

    dc = (reached / TOTAL_DATASETS) * 100

    ver_values = df[df["reached_viz"]]["ver_iter0"].dropna()
    ver_mean   = ver_values.mean()
    ver_std    = ver_values.std()
    ver_median = ver_values.median()

    cer = (not_converged / reached * 100) if reached > 0 else np.nan

    air_values = df[df["reached_viz"] & df["converged"]]["n_iterations"]
    air_mean   = air_values.mean()
    air_std    = air_values.std()

    return {
        "config": config_name,
        "total": TOTAL_DATASETS,
        "started": started,
        "not_started": TOTAL_DATASETS - started,
        "reached_viz": int(reached),
        "failed_before_viz": started - int(reached),
        "converged": int(converged),
        "not_converged": int(not_converged),
        "DC_%": round(dc, 1),
        "VER_mean_%": round(ver_mean, 1),
        "VER_std_%": round(ver_std, 1),
        "VER_median_%": round(ver_median, 1),
        "CER_%": round(cer, 1),
        "AIR_mean": round(air_mean, 2),
        "AIR_std": round(air_std, 2),
        "per_dataset_df": df,
    }

---
### Plotting-Funktionen

Die folgenden Funktionen generieren die vier Abbildungen für die Thesis. Alle Plots werden als SVG gespeichert.

- `plot_coverage`: Dataset Coverage + Fehleraufschlüsselung (gestapelter Balken)
- `plot_ver_boxplot`: VER (Iteration 0) als Box-/Strip-Plot
- `plot_iterations`: Iterationsverteilung als gestapeltes Balkendiagramm
- `plot_metrics_table`: Metriken-Übersicht als Heatmap-Tabelle

In [ ]:
def plot_ver_boxplot(per_ds: pd.DataFrame, save_dir: Path):
    """Abbildung 2: VER (Iteration 0) – Box-/Strip-Plot."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    ver_data = []
    for cfg in CONFIG_ORDER:
        sub = per_ds[(per_ds["config"] == cfg) & (per_ds["reached_viz"] == True)
                     & per_ds["ver_iter0"].notna()]
        ver_data.append(sub["ver_iter0"].values)

    bp = ax.boxplot(ver_data, positions=np.arange(len(CONFIG_ORDER)), widths=0.45,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", linewidth=2))

    for patch, cfg in zip(bp["boxes"], CONFIG_ORDER):
        patch.set_facecolor(COLORS[cfg])
        patch.set_alpha(0.7)

    rng = np.random.default_rng(42)
    for i, (cfg, vals) in enumerate(zip(CONFIG_ORDER, ver_data)):
        jitter = rng.uniform(-0.18, 0.18, size=len(vals))
        ax.scatter(i + jitter, vals, color=COLORS[cfg], s=30, alpha=0.85,
                   zorder=5, edgecolors="white", linewidths=0.4)

    ax.set_xticks(np.arange(len(CONFIG_ORDER)))
    ax.set_xticklabels(CONFIG_ORDER, fontsize=11)
    ax.set_ylabel("VER Iteration 0 [%]", fontsize=11)
    ax.set_ylim(-5, 105)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.axhline(0, color="gray", linestyle=":", linewidth=0.7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title("VER in Iteration 0 (Erstversuch) pro Konfiguration", fontsize=11, pad=10)

    plt.tight_layout()
    path = save_dir / "05-rq3-ver-boxplot.svg"
    plt.savefig(path, format="svg", bbox_inches="tight")
    plt.close()
    print(f"Gespeichert: {path}")

In [ ]:
def plot_iterations(per_ds: pd.DataFrame, save_dir: Path):
    """Abbildung 3: Iterationsverteilung (gestapelter Balken)."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    iter_colors = ["#009B91", "#7990AB", "#B2BFCE", "#546B86", "#334152"]
    max_iter = int(per_ds[per_ds["reached_viz"] == True]["n_iterations"].max())

    iter_data = {}
    for cfg in CONFIG_ORDER:
        sub = per_ds[(per_ds["config"] == cfg) & (per_ds["reached_viz"] == True)]
        counts = sub["n_iterations"].value_counts().sort_index()
        iter_data[cfg] = {k: counts.get(k, 0) for k in range(1, max_iter + 1)}

    x = np.arange(len(CONFIG_ORDER))
    width = 0.55
    bottoms = np.zeros(len(CONFIG_ORDER))

    for it in range(1, max_iter + 1):
        vals = [iter_data[c].get(it, 0) for c in CONFIG_ORDER]
        color = iter_colors[min(it - 1, len(iter_colors) - 1)]
        bars = ax.bar(x, vals, width, bottom=bottoms, color=color,
                      label=f"{it} Iteration{'en' if it > 1 else ''}",
                      edgecolor="white", linewidth=0.8)
        for j, (bar, v) in enumerate(zip(bars, vals)):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bottoms[j] + v / 2,
                        str(v), ha="center", va="center", fontsize=9,
                        fontweight="bold", color="black")
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(CONFIG_ORDER, fontsize=11)
    ax.set_ylabel("Anzahl Datensätze", fontsize=11)
    ax.set_ylim(0, max(bottoms) + 5)
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        fontsize=8.5,
        framealpha=0.9,
        borderaxespad=0,
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title("Verteilung benötigter Iterationen", fontsize=11, pad=10)

    plt.tight_layout()
    path = save_dir / "05-rq3-iterations.svg"
    plt.savefig(path, format="svg", bbox_inches="tight")
    plt.close()
    print(f"Gespeichert: {path}")

In [ ]:
def plot_coverage(summary: pd.DataFrame, save_dir: Path):
    """Abbildung 1: Dataset Coverage + Failure Breakdown (gestapelter Balken)."""
    fig, ax = plt.subplots(figsize=(8, 4.5))

    categories_labels = [
        "Daten nicht vorhanden",
        "Fehler beim Daten Laden",
        "Viz-Phase erreicht,\nnicht konvergiert",
        "Vollständig\nkonvergiert",
    ]
    colors_stack = ["#334152", "#546B86", "#B2BFCE", "#009B91"]

    data = {}
    for _, row in summary.iterrows():
        cfg = row["config"]
        not_started       = TOTAL_DATASETS - int(row["started"])
        failed_before_viz = int(row["started"]) - int(row["reached_viz"])
        not_conv          = int(row["reached_viz"]) - int(row["converged"])
        conv              = int(row["converged"])
        data[cfg] = [not_started, failed_before_viz, not_conv, conv]

    x = np.arange(len(CONFIG_ORDER))
    width = 0.55
    bottoms = np.zeros(len(CONFIG_ORDER))

    for i, (label, color) in enumerate(zip(categories_labels, colors_stack)):
        vals = [data[c][i] for c in CONFIG_ORDER]
        bars = ax.bar(x, vals, width, bottom=bottoms, color=color, label=label,
                      edgecolor="white", linewidth=0.8)
        for j, (bar, v) in enumerate(zip(bars, vals)):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, bottoms[j] + v / 2,
                        str(v), ha="center", va="center", fontsize=9,
                        fontweight="bold", color="black")
        bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(CONFIG_ORDER, fontsize=11)
    ax.set_ylabel("Anzahl Datensätze", fontsize=11)
    ax.set_ylim(0, 55)
    ax.set_yticks(range(0, 51, 10))
    ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="N=50 gesamt")
    ax.legend(
        loc="upper left",
        bbox_to_anchor=(1.02, 1),  # rechts außerhalb
        fontsize=8.5,
        framealpha=0.9,
        borderaxespad=0,
    )    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_title("Datensatzabdeckung nach Konfiguration (N=50)", fontsize=11, pad=10)

    plt.tight_layout()
    path = save_dir / "05-rq3-coverage.svg"
    plt.savefig(path, format="svg", bbox_inches="tight")
    plt.close()
    print(f"Gespeichert: {path}")

In [ ]:
def plot_metrics_table(summary: pd.DataFrame, save_dir: Path):
    """Abbildung 4: Metriken-Übersicht (Heatmap-Tabelle)."""
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.axis("off")

    metrics = ["DC [%]", "VER₀ MW [%]", "VER₀ SD [%]", "CER [%]", "AIR MW", "AIR SD"]
    table_data = []
    for cfg in CONFIG_ORDER:
        row = summary[summary["config"] == cfg].iloc[0]
        table_data.append([
            f"{row['DC_%']:.1f}",
            f"{row['VER_mean_%']:.1f}",
            f"{row['VER_std_%']:.1f}",
            f"{row['CER_%']:.1f}",
            f"{row['AIR_mean']:.2f}",
            f"{row['AIR_std']:.2f}",
        ])

    table = ax.table(
        cellText=table_data,
        rowLabels=CONFIG_ORDER,
        colLabels=metrics,
        cellLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    for j in range(len(metrics)):
        table[0, j].set_facecolor("#334152")
        table[0, j].set_text_props(color="white", fontweight="bold")

    for i, cfg in enumerate(CONFIG_ORDER):
        table[i + 1, -1].set_facecolor(COLORS[cfg])
        table[i + 1, -1].set_text_props(color="white", fontweight="bold")
        for j in range(len(metrics)):
            if (i + 1) % 2 == 0:
                table[i + 1, j].set_facecolor("#D9E5EC")

    ax.set_title("Tabelle: Metriken-Übersicht RQ-3 nach Konfiguration", fontsize=11, pad=12)

    plt.tight_layout()
    path = save_dir / "05-rq3-metrics-table.svg"
    plt.savefig(path, format="svg", bbox_inches="tight")
    plt.close()
    print(f"Gespeichert: {path}")

---
## Analyse und Auswertung

Im folgenden Abschnitt werden die Analysen ausgeführt und die Ergebnisse ausgegeben.

### Metriken pro Konfiguration

Für jede der vier Konfigurationen wird `analyse_config` aufgerufen. Die Ausgabe zeigt DC, VER, CER und AIR im Überblick.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
THESIS_FIG_DIR.mkdir(parents=True, exist_ok=True)

all_results = []
per_dataset_frames = {}

for cfg_name, cfg_path in CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Konfiguration: {cfg_name}")
    r = analyse_config(cfg_name, cfg_path)
    all_results.append({k: v for k, v in r.items() if k != "per_dataset_df"})
    per_dataset_frames[cfg_name] = r["per_dataset_df"]

    print(f"  Datensätze gesamt:          {r['total']}")
    print(f"  Davon gestartet:            {r['started']}")
    print(f"  Viz-Phase erreicht:         {r['reached_viz']}  → DC = {r['DC_%']} %")
    print(f"  Konvergiert:                {r['converged']}")
    print(f"  Nicht konvergiert:          {r['not_converged']}  → CER = {r['CER_%']} %")
    print(f"  VER (Iter 0):  Mittelwert = {r['VER_mean_%']} %  (SD = {r['VER_std_%']} %,  Median = {r['VER_median_%']} %)")
    print(f"  Ø Iterationen (konv.):     {r['AIR_mean']} ± {r['AIR_std']}")

summary_df = pd.DataFrame(all_results)

print("\n\n=== ZUSAMMENFASSUNG ===")
print(summary_df[[
    "config", "DC_%", "VER_mean_%", "VER_std_%",
    "CER_%", "AIR_mean", "AIR_std",
    "reached_viz", "converged"
]].to_string(index=False))

### Iterationsverteilung

Wie viele Iterationen benötigte der Agent typischerweise bis zur Konvergenz? Die folgende Ausgabe zeigt die Häufigkeitsverteilung der Iterationsanzahl pro Konfiguration – nur für Datensätze, bei denen die Visualisierungsphase erreicht wurde.

In [ ]:
for cfg_name, df in per_dataset_frames.items():
    reached = df[df["reached_viz"]]
    dist = reached["n_iterations"].value_counts().sort_index()
    print(f"\n{cfg_name}:")
    for k, v in dist.items():
        print(f"  {k} Iteration(en): {v} Datensätze")

### VER-Verteilung

Wie verteilen sich die Visualisierungsfehler in Iteration 0? Die folgende Auswertung gruppiert die VER-Werte in vier Bereiche: perfekt (0 %), niedrig (1–33 %), mittel (34–66 %) und hoch (67–100 %).

In [ ]:
for cfg_name, df in per_dataset_frames.items():
    reached = df[df["reached_viz"] & df["ver_iter0"].notna()].copy()
    bins = [0, 1, 34, 67, 100]
    labels = ["0 % (perfekt)", "1–33 %", "34–66 %", "67–100 %"]
    reached["ver_bin"] = pd.cut(reached["ver_iter0"], bins=bins, labels=labels, include_lowest=True)
    print(f"\n{cfg_name}:")
    print(reached["ver_bin"].value_counts().sort_index().to_string())

---
### Abbildungen generieren

Die vier Thesis-Abbildungen werden erzeugt und in `THESIS_FIG_DIR` gespeichert.

In [ ]:
combined = []
for cfg_name, df in per_dataset_frames.items():
    df = df.copy()
    df["config"] = cfg_name
    combined.append(df)
per_ds = pd.concat(combined)
    
plot_coverage(summary_df, THESIS_FIG_DIR)
plot_ver_boxplot(per_ds, THESIS_FIG_DIR)
plot_iterations(per_ds, THESIS_FIG_DIR)
plot_metrics_table(summary_df, THESIS_FIG_DIR)

print("\nAlle Abbildungen erzeugt.")
# summary_df, per_ds